# Causal feature engineering

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

This notebook is a software tutorial using bundled example bars or a small deterministic fixture. It is not a trading recommendation or a research performance claim.

This example uses the production alignment and normalization helpers on a tiny time index. A five-minute value becomes visible at its close, never during the still-forming candle.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from features.pipeline.normalization import rolling_zscore
from market_data.transforms.align import align_timeframes

m1_index = pd.date_range("2025-01-02 09:00", periods=16, freq="min", tz="UTC")
m5_index = pd.date_range("2025-01-02 09:00", periods=4, freq="5min", tz="UTC")
m1 = pd.DataFrame({"close": np.arange(100.0, 116.0)}, index=m1_index)
m5 = pd.DataFrame({"completed_close": [104.0, 109.0, 114.0, 119.0]}, index=m5_index)
aligned = align_timeframes(m1, {"M5": m5})
assert pd.isna(aligned.loc[m1_index[1], "M5_completed_close"])
assert aligned.loc[m1_index[5], "M5_completed_close"] == 104.0
aligned[["close", "M5_completed_close"]].head(8)

Figure: M1 close and the latest completed M5 close; the dashed marker identifies when the first M5 value becomes visible.

In [ ]:
features = pd.DataFrame({"return": m1["close"].pct_change().fillna(0.0)}, index=m1_index)
train_mask = pd.Series(m1_index < m1_index[11], index=m1_index)
normalized = rolling_zscore(features, window=5, train_mask=train_mask, mode="frozen_fold")
fig, ax = plt.subplots(figsize=(10, 3.5))
aligned["close"].plot(ax=ax, label="M1 close")
aligned["M5_completed_close"].plot(ax=ax, drawstyle="steps-post", label="latest completed M5 close")
ax.axvline(m1_index[5], color="gray", linestyle="--", label="first M5 close visible")
ax.set(title="Higher-timeframe values appear after completion", ylabel="Price points")
ax.legend()
plt.show()
print(
    f"train rows={int(train_mask.sum())}; validation rows={int((~train_mask).sum())}; normalized shape={normalized.shape}"
)

## Interpret the output

Forward-fill carries the last completed source candle across later M1 rows. The training mask is passed to the scaler; validation rows cannot fit new statistics. Real feature generation uses the same causal alignment contract and versioned cache identity.

See [`align_timeframes`](../../api/domains/data), [`rolling_zscore`](../../api/domains/data), [feature catalog](../../reference/feature_catalog), and [cache identity](../../reference/cache_and_schema_versions).